In [ ]:
# SETUP — install esam3, detect environment, resolve HF token (or skip if local).
import os
import subprocess
import sys
from pathlib import Path

import torch

# Install the repo from GitHub with the qlora + tensorboard extras.
# (Docker image is deferred — see GitHub issue #34.)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "git+https://github.com/NguyenJus/Efficient-SAM3-Finetuning.git#egg=efficient-sam3-finetuning[qlora,tensorboard]",
    ]
)

from esam3.notebook_helpers import (  # noqa: E402
    check_local_checkpoint,
    detect_env,
    resolve_hf_token,
)

env = detect_env()
assert torch.cuda.is_available(), (
    "No CUDA detected. In Colab: Runtime → Change runtime type → GPU. On RunPod: deploy a GPU pod."
)
local_present = check_local_checkpoint(Path("models/sam3.1"), "sam3.1_multiplex.pt")
token = resolve_hf_token(env, local_present)
if token is not None:
    os.environ["HF_TOKEN"] = token
print(
    f"mode: env={env}, local_checkpoint={local_present}, "
    f"hf_auth={'skipped' if token is None else 'enabled'}"
)

In [ ]:
# FORM — three values: dataset path, format, run name. v0 is text-only.
dataset_path: str = ""  #@param {type:"string"}
data_format: str = "coco"  #@param ["coco", "hf"]
run_name: str = "my-run"  #@param {type:"string"}

assert dataset_path, (
    "dataset_path is required. For coco: path to a folder with train/ and val/. For hf: an HF dataset id."  # noqa: E501
)

In [ ]:
# GENERATE — derive preset, load template, deep-merge user inputs, write config.yaml, run esam3.
import importlib.resources
import subprocess

import yaml

from esam3.presets import pick_preset, preset_label

patch = pick_preset()
template_name = (
    "coco_text_qlora.yaml" if patch["peft"]["method"] == "qlora" else "coco_text_lora.yaml"
)
template_text = (importlib.resources.files("esam3.cli.templates") / template_name).read_text()
config = yaml.safe_load(template_text)


def _deep_merge(dst: dict, src: dict) -> dict:
    for k, v in src.items():
        if isinstance(v, dict) and isinstance(dst.get(k), dict):
            _deep_merge(dst[k], v)
        else:
            dst[k] = v
    return dst


_deep_merge(config, patch)
config["run"]["name"] = run_name
config["data"]["format"] = data_format

# COCO: dataset_path is a directory containing train/ and val/ subdirectories.
# HF:   dataset_path is the HF dataset id, passed through as-is.
if data_format == "coco":
    _PREF = ("_annotations.coco.json", "instances.json", "annotations.json")

    def _resolve_split(split_dir: Path) -> str:
        if not split_dir.is_dir():
            raise FileNotFoundError(
                f"COCO split directory not found: {split_dir}. Expected layout: "
                f"<dataset_path>/train/ and <dataset_path>/val/."
            )
        for pref in _PREF:
            cand = split_dir / pref
            if cand.is_file():
                return str(cand)
        json_candidates = sorted(p for p in split_dir.glob("*.json"))
        if json_candidates:
            return str(json_candidates[0])
        raise FileNotFoundError(
            f"No COCO annotation JSON found under {split_dir}. "
            f"Tried (in order): {_PREF + ('*.json (lex sort)',)}."  # noqa: RUF005
        )

    dataset_root = Path(dataset_path)
    for split in ("train", "val"):
        split_dir = dataset_root / split
        config["data"][split]["annotations"] = _resolve_split(split_dir)
        config["data"][split]["images"] = str(split_dir)
else:  # hf
    # HF adapter receives the dataset id directly. The template's data.train/val
    # paths are placeholders the adapter ignores under format='hf'.
    config.setdefault("data", {})["hf"] = {"id": dataset_path}

with open("config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

os.environ["ESAM3_PRESET_LABEL"] = preset_label()

proc = subprocess.Popen(
    ["esam3", "run", "--config", "config.yaml"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True,
)
buffer: list[str] = []
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
    buffer.append(line)
rc = proc.wait()
if rc != 0:
    print("\n--- last 50 lines ---")
    print("".join(buffer[-50:]))
    raise SystemExit(rc)

In [ ]:
# RESULTS — render summary.md inline and display sample overlays.
import shutil
from pathlib import Path

from IPython.display import Image, Markdown, display

latest = max(Path("runs").iterdir(), key=lambda p: p.stat().st_mtime)
display(Markdown((latest / "summary.md").read_text()))
for png in sorted((latest / "samples").glob("*.png")):
    display(Image(filename=str(png)))

# Download:
if detect_env() == "colab":
    archive = Path(
        shutil.make_archive(str(latest), "zip", root_dir=latest.parent, base_dir=latest.name)
    )
    print(f"To download: from google.colab import files; files.download({str(archive)!r})")
else:
    print("RunPod: copy the run with")
    print(f"  scp -P <pod_port> root@<pod_host>:/workspace/{latest}.zip ./")
    print(f"(or zip first: shutil.make_archive('{latest}', 'zip'))")